<a href="https://colab.research.google.com/github/aravindasubramanian-hash/Capstone-Project-/blob/main/Gnani_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [177]:
# Step 1: mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
ROOT = '/content/drive/MyDrive/Capstone_Project_Dataset_FTTT_2026'
os.chdir(ROOT)
print(os.getcwd())
print(os.listdir())

/content/drive/MyDrive/Capstone_Project_Dataset_FTTT_2026
['CHANGELOG.md', 'DATA_DICTIONARY.md', 'ZIP_SCHEMA.md', 'README.md', 'validate_dataset.py', 'data', 'practice', 'tools', 'kb', 'step4_retrieval.py']


In [4]:
!python3 validate_dataset.py

PASS — all 9 CSVs, 21 KB documents, primary/foreign keys, order math,
       practice evidence, verification gates and company naming validated.
       1,000 orders | 1,783 items | 244 logs | 52 questions | 23 conversations


In [12]:
import csv
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Capstone_Project_Dataset_FTTT_2026")
def load(name):
    with open(ROOT / name, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

customers = load("data/customers.csv")
orders = load("data/orders.csv")
items = load("data/order_items.csv")
products = load("data/products.csv")
pincodes = load("data/pincodes.csv")
holidays = load("data/holidays.csv")
questions = load("practice/questions.csv")
conversations = load("practice/conversations.csv")

print(len(orders), len(items), len(questions), len(conversations))

1000 1783 52 96


In [13]:
import re
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Capstone_Project_Dataset_FTTT_2026")
def parse_front_matter(text):
    meta = {}
    if text.startswith("---"):
        block = text.split("---", 2)[1]
        for line in block.strip().splitlines():
            k, _, v = line.partition(":")
            meta[k.strip()] = v.strip()
    return meta

chunks = []
for path in sorted((ROOT / "kb").glob("*.md")):
    text = path.read_text(encoding="utf-8")
    meta = parse_front_matter(text)
    parts = re.split(r"^##\s*\[([^\]]+)\]\s*(.*)$", text, flags=re.M)
    # parts: [preamble, id, title, body, id, title, body, ...]
    for i in range(1, len(parts), 3):
        chunks.append({
            "section_id": parts[i],
            "title": parts[i + 1].strip(),
            "body": parts[i + 2].strip(),
            "doc_id": meta.get("doc_id"),
            "doc_type": meta.get("doc_type"),
            "status": meta.get("status"),
            "effective_from": meta.get("effective_from"),
        })

print(len(chunks), "sections")

84 sections


In [14]:
kb_path = ROOT / "kb"
print(f"KB path: {kb_path}")
print(f"KB exists: {kb_path.exists()}")
files = list(kb_path.glob("*.md")) if kb_path.exists() else []
print(f"MD files found: {len(files)}")
for f in files[:5]:
    print(f"  - {f.name}")

KB path: /content/drive/MyDrive/Capstone_Project_Dataset_FTTT_2026/kb
KB exists: True
MD files found: 21
  - 00_about_this_knowledge_base.md
  - agent_conversation_style.md
  - agent_escalation_handoff.md
  - agent_identity_verification.md
  - faq_general.md


In [15]:
!pip install rank-bm25

In [16]:
from rank_bm25 import BM25Okapi

corpus = [chunk['body'].lower().split() for chunk in chunks]
section_ids = [chunk['section_id'] for chunk in chunks]
bm25 = BM25Okapi(corpus)
print(f"BM25 index built with {len(corpus)} documents\n")

BM25 index built with 84 documents



In [17]:
hits = 0
total_with_gold = 0

for q in questions:
    gold_str = q.get('gold_evidence', '').strip()
    if not gold_str:
        continue

    total_with_gold += 1
    gold_sections = set(s.strip() for s in gold_str.split(';'))

    found = False
    for text_field in [q.get('question', ''), q.get('asr_text', '')]:
        if not text_field:
            continue

        query_tokens = text_field.lower().split()
        scores = bm25.get_scores(query_tokens)

        top5_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
        retrieved_sections = set(section_ids[i] for i in top5_indices)

        if gold_sections & retrieved_sections:
            hits += 1
            found = True
            break

    if total_with_gold % 10 == 0:
        print(f"Processed {total_with_gold} questions...")

print(f"\n=== Step 4 Results ===")
print(f"Retrieval Hit Rate: {hits}/{total_with_gold} = {100*hits/total_with_gold:.1f}%")

Processed 10 questions...
Processed 20 questions...
Processed 30 questions...
Processed 40 questions...

=== Step 4 Results ===
Retrieval Hit Rate: 24/47 = 51.1%


In [95]:
# ===== AGENT SETUP: loads data, BM25 retrieval, tools, Gemini LLM agent and Gnani voice helpers (run once after any restart) =====
!pip install -q rank-bm25 google-genai
import os, re, csv, json, time, sys, threading, requests
from pathlib import Path
from datetime import datetime, timedelta, date
from collections import defaultdict
from google.colab import drive, userdata
drive.mount('/content/drive')
from rank_bm25 import BM25Okapi
from google import genai
from google.genai import types

ROOT = Path("/content/drive/MyDrive/Capstone_Project_Dataset_FTTT_2026")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from tools import lookup_order, create_ticket

def load(name):
    with open(ROOT / name, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

customers = load("data/customers.csv"); orders = load("data/orders.csv"); items = load("data/order_items.csv")
products = load("data/products.csv"); pincodes = load("data/pincodes.csv"); holidays = load("data/holidays.csv")
questions = load("practice/questions.csv"); conversations = load("practice/conversations.csv")

def parse_front_matter(text):
    meta = {}
    if text.startswith("---"):
        block = text.split("---", 2)[1]
        for line in block.strip().splitlines():
            k, _, v = line.partition(":")
            meta[k.strip()] = v.strip()
    return meta

chunks = []
for path in sorted((ROOT / "kb").glob("*.md")):
    text = path.read_text(encoding="utf-8")
    meta = parse_front_matter(text)
    parts = re.split(r"^##\s*\[([^\]]+)\]\s*(.*)$", text, flags=re.M)
    for i in range(1, len(parts), 3):
        chunks.append({"section_id": parts[i], "title": parts[i + 1].strip(), "body": parts[i + 2].strip(),
                       "doc_id": meta.get("doc_id"), "doc_type": meta.get("doc_type"), "status": meta.get("status"), "effective_from": meta.get("effective_from")})
bm25 = BM25Okapi([c["body"].lower().split() for c in chunks])
SECTION = {c["section_id"]: c for c in chunks}
print(len(orders), len(items), len(questions), len(conversations), "|", len(chunks), "sections")

# ---- text helpers ----
WORDS = {"zero":"0","one":"1","two":"2","three":"3","four":"4","five":"5","six":"6","seven":"7","eight":"8","nine":"9","शून्य":"0","एक":"1","दो":"2","तीन":"3","चार":"4","पांच":"5","पाँच":"5","छह":"6","छः":"6","सात":"7","आठ":"8","नौ":"9"}

def normalize(msg):
    msg = msg.translate(str.maketrans("०१२३४५६७८९", "0123456789"))
    msg = re.sub(r"(?<=\d),(?=\d)", "", msg)
    out = []
    run_tok = []
    run_dig = []
    for t in msg.split():
        w = t.strip(",.।").lower()
        if w in WORDS:
            run_tok.append(t)
            run_dig.append(WORDS[w])
            continue
        if len(run_dig) >= 4:
            out.append("".join(run_dig))
        else:
            out.extend(run_tok)
        run_tok = []
        run_dig = []
        out.append(t)
    if len(run_dig) >= 4:
        out.append("".join(run_dig))
    else:
        out.extend(run_tok)
    return " ".join(out)

def find_order_id(msg):
    m = re.search(r"\b\d{4}\s?\d{4}\b", msg)
    if m:
        return re.sub(r"\D", "", m.group()), m.group()
    return None, None

def find_proof(msg):
    m6 = re.search(r"\b\d{6}\b", msg)
    if m6:
        return m6.group()
    m4 = re.search(r"\b\d{4}\b", msg)
    if m4:
        return m4.group()
    return None

# ---- policy arithmetic (computed in code, not by the LLM) ----
TODAY = date(2026, 10, 20)
HOLIDAYS = set(h["date"] for h in holidays)
MAX_DAYS = {"upi": 3, "card": 7, "netbanking": 5, "wallet": 0, "cod_wallet": 0, "cod_bank": 7}
REFUND_WORDS = ["refund", "रिफंड", "रीफंड", "paisa", "paise", "पैसे"]

def business_days_since(start):
    d = start
    n = 0
    while d < TODAY:
        d += timedelta(days=1)
        if d.weekday() != 6 and d.isoformat() not in HOLIDAYS:
            n += 1
    return n

def refund_overdue(order):
    for p in ("cancellation", "rto", "return"):
        started = order.get(p + "_refund_initiated_on")
        method = order.get(p + "_refund_method")
        if not started or method not in MAX_DAYS:
            continue
        if p == "return" and order.get("return_refund_completed_on"):
            continue
        if business_days_since(datetime.strptime(started, "%Y-%m-%d").date()) > MAX_DAYS[method] + 3:
            return True
    return False

def damaged_value(order):
    sku = order.get("damage_sku")
    if not sku:
        return 0
    return sum((i["unit_price_inr"] or 0) * (i["qty"] or 1) for i in order.get("items", []) if i["sku"] == sku)

RET_DAYS = {"fashion": 15, "electronics": 7, "home_kitchen": 10, "beauty_sealed": 7, "books_media": 10}
def return_info(order):
    oid = str(order.get("order_id") or order.get("id") or "")
    d = str(order.get("delivered_on") or "")[:10]
    placed = str(order.get("placed_at") or "")[:10]
    out = []
    for it in items:
        if str(it.get("order_id")) != oid:
            continue
        cat = it.get("return_category")
        days = RET_DAYS.get(cat)
        if cat == "fashion" and "2026-10-01" <= placed <= "2026-10-15":
            days = 30
        if cat == "electronics" and d and d < "2026-08-01":
            days = 10
        row = {"item": it.get("title"), "category": cat, "return_window_days": days}
        if days is not None and d:
            row["return_deadline"] = str(date.fromisoformat(d) + timedelta(days=days))
        if cat == "fashion" and not order.get("is_plus"):
            row["reverse_pickup_fee_inr_if_change_of_mind"] = 49
        out.append(row)
    return out

def facts(order):
    if not order:
        return None
    contacts = order.get("contacts_last_7_days") or 0
    return {"refund_overdue_per_ESC-1": refund_overdue(order), "repeat_contact_rule_triggered": contacts >= 2,
            "damaged_item_value_inr": damaged_value(order),
            "compensation_allowed_for_late_delivery": order.get("delivery_type") == "express" and not order.get("is_plus"),
            "return_info": return_info(order)}

# ---- template replies (verification flow) ----
def compose_reply(decision, state, msg, lang):
    hi = lang in ("hi", "hi-en")
    if decision == "clarify":
        if state["oid"] is None:
            return "नमस्ते, मैं ZentroMart का AI असिस्टेंट हूँ। कृपया अपना ऑर्डर नंबर बताइए।" if hi else "Hello, I'm ZentroMart's AI assistant. Please share your order ID."
        return "पहचान के लिए अपने रजिस्टर्ड फ़ोन के आखिरी चार अंक या डिलीवरी पिनकोड बताइए।" if hi else "To verify you, please tell me the last 4 digits of your registered phone number or your delivery pincode."
    return "ठीक है।" if hi else "Okay."

def compose_reply2(decision, state, msg, lang):
    hi = lang in ("hi", "hi-en")
    if decision == "clarify" and state["oid"] is not None and state["attempts"] >= 1:
        return ("आपका जवाब हमारे रिकॉर्ड से मेल नहीं खाता। कृपया फिर से कोशिश करें: रजिस्टर्ड फ़ोन के आखिरी चार अंक या डिलीवरी पिनकोड बताइए।" if hi else "That doesn't match our records. Please try again with the last 4 digits of your registered phone number or your delivery pincode.")
    if decision == "refuse" and state["attempts"] >= 2:
        return ("मैं आपकी पहचान की पुष्टि नहीं कर पाया, इसलिए रजिस्टर्ड जानकारी साझा नहीं कर सकता। कृपया My Orders देखें या बाद में फिर संपर्क करें।" if hi else "I couldn't verify you, so I can't share the registered order details. Please check My Orders in the app, or contact us again later.")
    if decision == "answer" and state.get("order"):
        o = state["order"]
        parts = []
        for k, label in (("placed_at", "placed"), ("shipped_on", "shipped"), ("promised_date", "promised for"), ("delivered_on", "delivered")):
            if o.get(k):
                parts.append(label + " " + str(o[k])[:10])
        info = ", ".join(parts)
        if hi:
            return "धन्यवाद, पहचान हो गई। ऑर्डर की स्थिति: " + str(o.get("status")) + " (" + info + ")। मैं और क्या मदद करूँ?"
        return "Thank you, you're verified. Your order is " + str(o.get("status")) + " (" + info + "). How can I help?"
    return compose_reply(decision, state, msg, lang)

# ---- Gemini ----
def get_secret(*names):
    for n in names:
        try:
            return userdata.get(n)
        except Exception:
            pass
    return None

client = genai.Client(api_key=get_secret("Gemini_API_key", "GEMINI_API_KEY"), http_options=types.HttpOptions(timeout=30000))
MODEL = "gemini-3.1-flash-lite"

SYSTEM = ("You are ZentroMart's customer support AI assistant for an Indian online store. Today is 20 October 2026.\n"
"Choose ONE decision for the customer's latest message:\n"
"- answer: you can fully answer using the policy sections and verified order data provided.\n"
"- clarify: you need more information. If the customer asks about an order but no verified order data is provided, ask only for the order ID (in the first message, introduce yourself as an AI assistant).\n"
"- escalate: hand over to a human. ESC-1 conditions: damaged/defective/wrong item worth more than Rs 10,000; activity the customer did not make; legal action, consumer court or police; third or later contact within 7 days (contacts_last_7_days >= 2); refund more than 3 business days past its maximum timeline (business days skip Sundays and holidays); customer asks for a human; product safety risk.\n"
"- refuse: the request is not allowed by policy (e.g. cancelling a shipped order, compensation the policy does not allow). Explain briefly and give the allowed alternatives.\n"
"Rules: use only the provided sections and order data. If documents disagree, prefer status current and the newest effective_from. Never invent facts. Never reveal phone numbers or pincodes. If a refund has not arrived, share its RRN. "
"Reply in the customer's language and script (hi = Devanagari Hindi, hi-en = Hinglish in Latin script, en = English). Keep the reply under 60 words, friendly and easy to say aloud. Write dates like 27 October 2026.\n"
"Extra rules: Trust the computed facts. If repeat_contact_rule_triggered or refund_overdue_per_ESC-1 is true, the decision must be escalate. "
"If compensation_allowed_for_late_delivery is false and the customer asks for a voucher or compensation for a late delivery, the decision is refuse. "
"When refusing, start by apologising, then give each allowed alternative with its time limit in days. "
"Always state the concrete dates (for example a return deadline = delivered date + the return window) and day counts from the policy, and mention named steps such as the quality check. "
"Use return_info from the computed facts exactly for return windows and deadlines; never calculate them yourself, and quote the deadline date and the number of days. If an item has a reverse pickup fee of 49, mention the Rs 49 fee (waived for Plus members or damaged items).\n"
"If asked where a refund for a returned item is, mention the quality check (QC) at the warehouse and that the refund starts within 2 business days of pickup (Plus members: at pickup).\n"
"If asked when an order was placed, shipped or delivered, state the exact date from the order data.\n"
"If the customer asks to cancel an order that has already shipped, say you have to decline the cancellation, then offer both options: refuse the parcel at the door (a prepaid refund arrives 7 to 10 calendar days after the last delivery attempt) or return the item within its return window (give the days).\n"
"If the customer asks what happens when they are not at home, say the courier makes up to 3 delivery attempts and sends an SMS to reschedule after a failed attempt.\n"
"When escalating, say a ticket has been created and a human colleague will contact the customer.\n"
"Output JSON only: {\"intent\": \"short_snake_case\", \"decision\": \"answer|clarify|escalate|refuse\", \"reply\": \"...\"}")

FALLBACK_MODELS = ["gemini-flash-lite-latest", "gemini-flash-latest"]
def gem(prompt, system=None, as_json=False):
    cfg = types.GenerateContentConfig(system_instruction=system, temperature=0.2, response_mime_type=("application/json" if as_json else None))
    last = ""
    for attempt in range(6):
        mdl = MODEL if attempt < 3 else FALLBACK_MODELS[(attempt - 3) % len(FALLBACK_MODELS)]
        try:
            return client.models.generate_content(model=mdl, contents=prompt, config=cfg).text
        except Exception as e:
            last = str(e)[:160]
            time.sleep(2 * (attempt + 1))
    print("gem failed:", last)
    return None

DROP = {"delivery_pincode", "customer_id", "found", "verified"}

def order_view(order):
    return {k: v for k, v in order.items() if k not in DROP and v is not None}

def retrieve(msg, lang):
    q = msg
    if lang != "en":
        q = gem("Rewrite this customer message as a short English keyword query for searching support policies. Output only the query.\nMessage: " + msg) or msg
    scores = bm25.get_scores(q.lower().split())
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:6]
    ids = [chunks[i]["section_id"] for i in top]
    low = (msg + " " + q).lower()
    boosts = [(r"not at home|not home|nobody|not available|ghar (par|pe) nahi|घर पर नहीं|घर पे नहीं", ["DEL-3"]),
              (r"cancel|कैंसिल|रद्द|रद", ["CAN-2", "REF-1"]),
              (r"return|वापस|लौटा", ["RET-2", "RET-6", "RET-8"]),
              (r"refund|रिफ़ंड|रिफंड|पैसे|paise", ["REF-1", "RET-7"])]
    for pat, secs in boosts:
        if re.search(pat, low):
            for s in secs:
                if s in SECTION and s not in ids:
                    ids.append(s)
    if "ESC-1" not in ids:
        ids.append("ESC-1")
    return ids

def ctx_text(ids):
    parts = []
    for i in ids:
        c = SECTION[i]
        parts.append("[" + c["section_id"] + "] " + c["title"] + " (status " + str(c["status"]) + ", effective " + str(c["effective_from"]) + ")\n" + c["body"])
    return "\n\n".join(parts)

def llm_decide(state, msg, lang):
    ids = retrieve(msg, lang)
    order = order_view(state["order"]) if state.get("order") else None
    prompt = ("Policy sections:\n" + ctx_text(ids) + "\n\nVerified order data (null if none):\n" + json.dumps(order, ensure_ascii=False, default=str) +
              "\n\nComputed facts (trust these):\n" + json.dumps(facts(state.get("order")), ensure_ascii=False) +
              "\n\nConversation so far:\n" + "\n".join(state["history"][-6:]) + "\n\nCustomer language code: " + lang + "\nLatest customer message: " + msg)
    raw = gem(prompt, SYSTEM, as_json=True)
    try:
        out = json.loads(raw)
        if isinstance(out, list):
            out = out[0]
    except Exception:
        out = {"intent": "parse_error", "decision": "escalate", "reply": ""}
    d = str(out.get("decision", "")).lower().strip()
    if d not in ("answer", "clarify", "escalate", "refuse"):
        d = "escalate"
    out["decision"] = d
    out["sections"] = ids
    return out

ESC_SUFFIX = {"en": " I've created a ticket and a human colleague will contact you.",
              "hi": " मैंने एक टिकट बना दिया है और हमारा सहयोगी आपसे संपर्क करेगा।",
              "hi-en": " Maine ek ticket bana diya hai aur hamare colleague aapse sampark karenge."}

def new_state():
    return {"oid": None, "verified": False, "attempts": 0, "history": []}

def agent_turn_llm(state, msg, lang):
    state.setdefault("history", [])
    msg = normalize(msg)
    calls = []
    oid, raw = find_order_id(msg)
    rest = msg.replace(raw, " ") if raw else msg
    if oid:
        state["oid"] = oid
    state["history"].append("customer: " + msg)

    if state["oid"] is not None and not state["verified"]:
        calls.append("lookup_order")
        proof = find_proof(rest)
        if proof is None:
            decision = "clarify"
        else:
            res = lookup_order(state["oid"], proof)
            if res.get("verified"):
                state["verified"] = True
                state["order"] = res
                decision = "answer"
            else:
                state["attempts"] += 1
                decision = "refuse" if state["attempts"] >= 2 else "clarify"
        reply = compose_reply2(decision, state, msg, lang)
        state["history"].append("agent: " + reply)
        return decision, calls, reply

    out = llm_decide(state, msg, lang)
    decision = out["decision"]
    reply = out.get("reply") or ""
    f = facts(state.get("order"))
    if f:
        low = msg.lower() + " " + str(out.get("intent")).lower()
        if f["repeat_contact_rule_triggered"]:
            decision = "escalate"
        if f["refund_overdue_per_ESC-1"] and ("refund" in low or any(w in low for w in REFUND_WORDS)):
            decision = "escalate"
    if decision == "escalate":
        summary = ("Issue: " + msg[:120] + " | Intent: " + str(out.get("intent")) + " | Order: " + str(state["oid"]) +
                   " | Customer said: " + " / ".join(state["history"][-4:])[:200] + " | Checked: identity verified via lookup_order, sections " + ",".join(out["sections"]) +
                   " | Escalation reason: " + str(out.get("intent")))
        t = create_ticket(summary, order_id=state["oid"], reason=str(out.get("intent")), intent=str(out.get("intent")))
        calls.append("create_ticket")
        state["ticket"] = t.get("ticket_id") if t.get("created") else None
        if state["ticket"]:
            reply = reply + ESC_SUFFIX.get(lang, ESC_SUFFIX["en"])
        else:
            reply = "I'm trying to connect you to a human colleague."
    state["history"].append("agent: " + reply)
    return decision, calls, reply

# ---- Gnani voice ----
GNANI_KEY = get_secret("GNANI_API_KEY")
TTS_LANG = {"en": "en-IN", "hi": "hi-IN", "hi-en": "hi-en"}
STT_LANG = {"en": "en-IN", "hi": "hi-IN", "hi-en": "hi-IN"}

def tts(text, lang, path):
    r = requests.post("https://api.vachana.ai/api/v1/tts/inference", headers={"Content-Type": "application/json", "X-API-Key-ID": GNANI_KEY},
                      json={"text": text, "model": "timbre-v2.5", "voice": "Nalini", "language": TTS_LANG.get(lang, "auto"), "speed": 1.0,
                            "audio_config": {"sample_rate": 16000, "num_channels": 1, "sample_width": 2, "encoding": "linear_pcm", "container": "wav"}})
    r.raise_for_status()
    open(path, "wb").write(r.content)
    return path

def stt(path, lang):
    with open(path, "rb") as f:
        r = requests.post("https://api.vachana.ai/stt/v3", headers={"X-API-Key-ID": GNANI_KEY}, files={"audio_file": f}, data={"language_code": STT_LANG.get(lang, "hi-IN")})
    r.raise_for_status()
    return r.json()["transcript"]

# English speech: the en-IN model can cut off long digit strings, so also try hi-IN (code-mixed) and keep the fuller transcript
_stt_raw = stt
def stt(path, lang):
    t = _stt_raw(path, lang)
    if lang == "en":
        try:
            t2 = _stt_raw(path, "hi")
            if len(re.findall(r"\d", t2)) > len(re.findall(r"\d", t)) or len(t2.split()) > len(t.split()) + 2:
                return t2
        except Exception:
            pass
    return t

print("setup complete | gemini key:", get_secret("Gemini_API_key") is not None, "| gnani key:", GNANI_KEY is not None)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1000 1783 52 96 | 84 sections
setup complete | gemini key: True | gnani key: True


In [36]:
# Gnani voice API check: text to speech
import requests
from google.colab import userdata
from IPython.display import Audio, display

KEY = userdata.get("GNANI_API_KEY")

r = requests.post(
    "https://api.vachana.ai/api/v1/tts/inference",
    headers={"Content-Type": "application/json", "X-API-Key-ID": KEY},
    json={
        "text": "नमस्ते, आपका ऑर्डर शिप हो चुका है।",
        "voice": "Nalini",
        "model": "timbre-v2.5",
        "language": "hi-IN",
        "speed": 1.0,
        "audio_config": {"sample_rate": 16000, "num_channels": 1, "sample_width": 2, "encoding": "linear_pcm", "container": "wav"},
    },
)
print("TTS status:", r.status_code, r.headers.get("Content-Type"))

if r.status_code == 200:
    open("/content/tts_test.wav", "wb").write(r.content)
    display(Audio("/content/tts_test.wav"))
else:
    print(r.text[:400])

TTS status: 200 audio/wav


In [37]:
with open("/content/tts_test.wav", "rb") as f:
    s = requests.post(
        "https://api.vachana.ai/stt/v3",
        headers={"X-API-Key-ID": KEY},
        files={"audio_file": f},
        data={"language_code": "hi-IN"},
    )
print("STT status:", s.status_code)
print(s.text[:500])

STT status: 200
{"success":true,"request_id":"01a172a3-cc78-77f4-bc1b-043571f62def","transcript":"नमस्ते, आपका ऑर्डर शिप हो चुका है।","model":"gnani-prisma-v2.5","processing_time":0.13045549392700195,"end_to_end_latency":0.3386719226837158,"output":{"literal":"नमस्ते, आपका ऑर्डर शिप हो चुका है।"}}


In [38]:
import requests
from google.colab import userdata
KEY = userdata.get("GNANI_API_KEY")

TTS_LANG = {"en": "en-IN", "hi": "hi-IN", "hi-en": "hi-en"}
STT_LANG = {"en": "en-IN", "hi": "hi-IN", "hi-en": "hi-IN"}

def tts(text, lang, path):
    r = requests.post(
        "https://api.vachana.ai/api/v1/tts/inference",
        headers={"Content-Type": "application/json", "X-API-Key-ID": KEY},
        json={"text": text, "model": "timbre-v2.5", "voice": "Nalini", "language": TTS_LANG.get(lang, "auto"), "speed": 1.0,
              "audio_config": {"sample_rate": 16000, "num_channels": 1, "sample_width": 2, "encoding": "linear_pcm", "container": "wav"}},
    )
    r.raise_for_status()
    open(path, "wb").write(r.content)
    return path

def stt(path, lang):
    with open(path, "rb") as f:
        r = requests.post("https://api.vachana.ai/stt/v3", headers={"X-API-Key-ID": KEY},
                          files={"audio_file": f}, data={"language_code": STT_LANG.get(lang, "hi-IN")})
    r.raise_for_status()
    return r.json()["transcript"]

In [51]:
# TEST 1: text sanity check of the Gemini agent
st = new_state()
for msg in ["Hello, I need help with my order", "Order number is 8978 2837", "Last four digits 4533"]:
    d, c, rp = agent_turn_llm(st, msg, "en")
    print(d, c, "|", rp[:110])

# TEST 2: full voice loop on the first Hindi conversation (customer TTS -> STT -> agent -> reply TTS)
from IPython.display import Audio, display
cid = conversations[0]["conversation_id"]
rows_v = sorted([r for r in conversations if r["conversation_id"] == cid], key=lambda r: int(r["turn"]))
sv = new_state()
last_reply = ""
for r in rows_v:
    lang = r["language"]
    tts(r["customer_message"], lang, "/content/in.wav")
    heard = stt("/content/in.wav", lang)
    d, c, last_reply = agent_turn_llm(sv, heard, lang)
    ok = "OK " if d == r["expected_decision"] else "MISS"
    print(ok, "turn", r["turn"], "| heard:", heard[:55])
    print("      ", d, "(want", r["expected_decision"] + ")", c, "|", last_reply[:90])
tts(last_reply, rows_v[-1]["language"], "/content/out.wav")
display(Audio("/content/out.wav"))


clarify [] | Hello! I am your ZentroMart AI assistant. I would be happy to help you with your order. Could you please provi
clarify ['lookup_order'] | To verify you, please tell me the last 4 digits of your registered phone number or your delivery pincode.
answer ['lookup_order'] | Thank you, you're verified. Your order is delivered (placed 2026-10-05, shipped 2026-10-06, promised for 2026-
OK  turn 1 | heard: हेलो, मुझे ऑर्डर के बारे में हेल्प चाहिए।
       clarify (want clarify) [] | नमस्ते! मैं ZentroMart का AI सपोर्ट असिस्टेंट हूँ। आपकी सहायता के लिए, क्या आप कृपया अपना 
OK  turn 2 | heard: ऑर्डर नंबर है 89782837
       clarify (want clarify) ['lookup_order'] | पहचान के लिए अपने रजिस्टर्ड फ़ोन के आखिरी चार अंक या डिलीवरी पिनकोड बताइए।
OK  turn 3 | heard: लास्ट फोर डिजिट्स 4,533 है
       answer (want answer) ['lookup_order'] | धन्यवाद, पहचान हो गई। ऑर्डर की स्थिति: delivered (placed 2026-10-05, shipped 2026-10-06, p
OK  turn 4 | heard: मैंने जेंट्रो विस्टा 43 इंच फोर के स्मार्ट टी 

In [78]:
# Step: evaluate the agent on all 96 turns of the 23 practice conversations (runs in background, about 10 min)
convs = defaultdict(list)
for r in conversations:
    convs[r["conversation_id"]].append(r)
import threading
MODEL = "gemini-3.1-flash-lite"

def check_must(reply, must_in, must_not):
    low = reply.lower()
    missing = []
    for fact in [f.strip() for f in (must_in or "").split(" ; ") if f.strip()]:
        variants = []
        for a in [a.strip() for a in fact.split(" / ") if a.strip()]:
            variants.append(a.lower())
            try:
                d = datetime.strptime(a, "%Y-%m-%d")
                variants.append((str(d.day) + " " + d.strftime("%B %Y")).lower())
                variants.append((d.strftime("%B") + " " + str(d.day)).lower())
                variants.append((str(d.day) + " " + d.strftime("%B")).lower())
            except ValueError:
                pass
        if not any(v in low for v in variants):
            missing.append(fact)
    bad = [p.strip() for p in (must_not or "").split(" ; ") if p.strip() and p.strip().lower() in low]
    return missing, bad

EVAL = {"rows": [], "done": False, "err": None, "t0": time.time()}

def run_eval():
    try:
        for cid, rws in convs.items():
            rws = sorted(rws, key=lambda x: int(x["turn"]))
            st = {"oid": None, "verified": False, "attempts": 0, "history": []}
            for r in rws:
                d, calls, reply = agent_turn_llm(st, r["customer_message"], r["language"])
                want = set(re.findall(r"(\w+)\(", r["expected_tool_calls"]))
                missing, bad = check_must(reply, r["must_include"], r["must_not_include"])
                EVAL["rows"].append({"cid": cid, "turn": r["turn"], "got": d, "want": r["expected_decision"], "calls": sorted(calls), "want_calls": sorted(want), "missing": missing, "bad": bad, "reply": reply, "msg": r["customer_message"]})
    except Exception as e:
        EVAL["err"] = repr(e)
    EVAL["done"] = True

threading.Thread(target=run_eval, daemon=True).start()
print("evaluation started in background")


evaluation started in background


In [91]:
# Step: evaluation report (run after the evaluation cell finishes; shows decision / tool-call / reply-fact scores and any misses)
R = EVAL["rows"]
print("rows", len(R), "done", EVAL["done"], "err", EVAL["err"])
print("decision", sum(r["got"]==r["want"] for r in R), "tools", sum(r["calls"]==r["want_calls"] for r in R), "facts ok", sum((not r["missing"]) and (not r["bad"]) for r in R))
for r in R:
    if r["got"]!=r["want"] or r["calls"]!=r["want_calls"]:
        print("DEC/TOOL MISS", r["cid"], r["turn"], r["got"], r["want"], r["calls"], r["want_calls"])
    if r["missing"] or r["bad"]:
        print("FACT", r["cid"], r["turn"], "MISSING", r["missing"], "BAD", r["bad"], "|", r["reply"][:260])

rows 96 done True err None
decision 96 tools 96 facts ok 96


In [96]:
# Step: test with my own recorded voice (STT -> agent -> TTS reply)
from IPython.display import Audio, display
import os
for path, lang in [("/content/order_help.wav", "en"), ("/content/order_help_hi.wav", "hi")]:
    st = new_state()
    heard = stt(path, lang)
    d, c, rp = agent_turn_llm(st, heard, lang)
    print("FILE", path)
    print("  heard   :", heard)
    print("  decision:", d, c)
    print("  reply   :", rp)
    tts(rp, lang, "/content/reply_" + os.path.basename(path))
    display(Audio("/content/reply_" + os.path.basename(path)))

FILE /content/order_help.wav
  heard   : हाय आई नीड हेल्प विथ माइ ऑर्डर दी ऑर्डर नंबर इज़ 15911102 एन्ड माइ डिलीवरी पिन कोड इज़ 440010
  decision: answer ['lookup_order']
  reply   : Thank you, you're verified. Your order is shipped (placed 2026-10-17, shipped 2026-10-20, promised for 2026-10-29). How can I help?


FILE /content/order_help_hi.wav
  heard   : नमस्ते मुझे अपने ऑर्डर के बारे में मदद चाहिए ऑर्डर नंबर है 15911102 और 440010
  decision: answer ['lookup_order']
  reply   : धन्यवाद, पहचान हो गई। ऑर्डर की स्थिति: shipped (placed 2026-10-17, shipped 2026-10-20, promised for 2026-10-29)। मैं और क्या मदद करूँ?


In [102]:
# Step: retrieval comparison - BM25 vs dense (Gemini embeddings) vs hybrid, same 47 gold-labelled questions, top-5, same protocol as step4_retrieval.py
import numpy as np, threading, re, time
from concurrent.futures import ThreadPoolExecutor
RET = {"done": False, "err": None, "res": None}
EMB_MODEL = "gemini-embedding-001"
def embed(texts, task):
    out = []
    for i in range(0, len(texts), 50):
        for attempt in range(5):
            try:
                r = client.models.embed_content(model=EMB_MODEL, contents=texts[i:i+50], config=types.EmbedContentConfig(task_type=task))
                out += [e.values for e in r.embeddings]
                break
            except Exception as e:
                last = e
                time.sleep(3 * (attempt + 1))
        else:
            raise last
    m = np.array(out, dtype=float)
    return m / np.linalg.norm(m, axis=1, keepdims=True)

SEC_IDS = [c["section_id"] for c in chunks]
SEC_STATUS = {c["section_id"]: c["status"] for c in chunks}
def run_retrieval_eval():
    try:
        D = embed([c["title"] + " (status " + c["status"] + ", effective " + str(c["effective_from"]) + ")\n" + c["body"] for c in chunks], "RETRIEVAL_DOCUMENT")
        qs = [q for q in questions if q["gold_evidence"].strip()]
        gold = {q["id"]: set(re.findall(r"[A-Z]+(?:v\d+)?-\d+", q["gold_evidence"])) & set(SEC_IDS) for q in qs}
        fields = {}
        for q in qs:
            for f in ("question", "asr_text"):
                fields[(q["id"], f)] = q[f]
        keys = list(fields)
        def rewrite(k):
            t = fields[k]
            lang = next(q["language"] for q in qs if q["id"] == k[0])
            if lang == "en" or not t:
                return t
            return gem("Rewrite this customer message as a short English keyword query for searching support policies. Output only the query.\nMessage: " + t) or t
        with ThreadPoolExecutor(6) as ex:
            rew = dict(zip(keys, ex.map(rewrite, keys)))
        Q = embed([fields[k] or " " for k in keys], "RETRIEVAL_QUERY")
        qv = dict(zip(keys, Q))
        def bm(text):
            s = bm25.get_scores(text.lower().split())
            return [SEC_IDS[i] for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)]
        def dense(k):
            s = D @ qv[k]
            return [SEC_IDS[i] for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)]
        def rrf(*rs):
            sc = {}
            for r in rs:
                for rank, sid in enumerate(r):
                    sc[sid] = sc.get(sid, 0) + 1.0 / (60 + rank + 1)
            return sorted(sc, key=lambda s: -sc[s])
        methods = {
            "bm25": lambda k: bm(fields[k]),
            "bm25_rewrite": lambda k: bm(rew[k]),
            "dense": lambda k: dense(k),
            "hybrid": lambda k: rrf(bm(fields[k]), dense(k)),
            "hybrid_rewrite": lambda k: rrf(bm(rew[k]), dense(k)),
        }
        res = {}
        for name, fn in methods.items():
            per = {}
            for q in qs:
                ranks = {f: fn((q["id"], f)) for f in ("question", "asr_text")}
                hit = {f: bool(gold[q["id"]] & set(ranks[f][:5])) for f in ranks}
                rr = 0.0
                for pos, sid in enumerate(ranks["question"]):
                    if sid in gold[q["id"]]:
                        rr = 1.0 / (pos + 1)
                        break
                stale = any(SEC_STATUS[s] == "superseded" for s in ranks["question"][:5]) and not any(SEC_STATUS[g] == "superseded" for g in gold[q["id"]])
                per[q["id"]] = {"any": hit["question"] or hit["asr_text"], "question": hit["question"], "asr": hit["asr_text"], "mrr": rr, "stale": stale, "top5_q": ranks["question"][:5]}
            n = len(qs)
            res[name] = {"n": n, "hit5_any": sum(p["any"] for p in per.values()), "hit5_question": sum(p["question"] for p in per.values()), "hit5_asr": sum(p["asr"] for p in per.values()), "mrr_question": round(sum(p["mrr"] for p in per.values()) / n, 3), "stale_in_top5": sum(p["stale"] for p in per.values()), "per": per}
        RET["res"] = res
        RET["gold"] = gold
    except Exception as e:
        RET["err"] = repr(e)
    RET["done"] = True
threading.Thread(target=run_retrieval_eval, daemon=True).start()
print("retrieval comparison started")


retrieval comparison started


In [113]:
# Step: Step 1 (LLM only) vs Step 2 (RAG) on practice/questions.csv, plus dense retrieval with superseded sections removed
import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor
import json, hashlib
EMB_CACHE_PATH = "/content/emb_cache.json"
try:
    EMB_CACHE = json.load(open(EMB_CACHE_PATH))
except Exception:
    EMB_CACHE = {}
def embed(texts, task):
    keys = [task + "|" + hashlib.md5(t.encode()).hexdigest() for t in texts]
    todo = [i for i, k in enumerate(keys) if k not in EMB_CACHE]
    for s in range(0, len(todo), 40):
        batch = todo[s:s + 40]
        for attempt in range(10):
            try:
                r = client.models.embed_content(model=EMB_MODEL, contents=[texts[i] for i in batch], config=types.EmbedContentConfig(task_type=task))
                for i, e in zip(batch, r.embeddings):
                    EMB_CACHE[keys[i]] = list(e.values)
                break
            except Exception as e:
                time.sleep(30 if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) else 3 * (attempt + 1))
        else:
            raise RuntimeError("embedding failed")
        json.dump(EMB_CACHE, open(EMB_CACHE_PATH, "w"))
        if s + 40 < len(todo):
            time.sleep(20)
    m = np.array([EMB_CACHE[k] for k in keys], dtype=float)
    return m / np.linalg.norm(m, axis=1, keepdims=True)

STEP = {"done": False, "err": None}
def parse_json(raw):
    try:
        o = json.loads(raw)
        if isinstance(o, list):
            o = o[0]
        return o
    except Exception:
        return {"decision": "parse_error", "reply": raw or ""}
def numbers(text):
    return set(n.rstrip(".,").replace(",", "") for n in re.findall(r"\d[\d,]*\.?\d*", text or ""))
BASE = ("You are ZentroMart's customer support AI assistant for an Indian online store. Today is 20 October 2026. "
        "Reply briefly in the customer's language (code: {lang}). Choose ONE decision: answer, clarify, escalate or refuse. "
        "Output JSON only: {{\"decision\": \"answer|clarify|escalate|refuse\", \"reply\": \"...\"}}.")
RAG_ADD = (" Answer only from the policy sections provided. If they do not cover the question, say you do not have that information "
           "and offer to connect a human colleague (decision escalate). Never invent policies, amounts or dates.")
def run_step_eval():
    try:
        D2 = embed([c["title"] + " (status " + c["status"] + ", effective " + str(c["effective_from"]) + ")\n" + c["body"] for c in chunks], "RETRIEVAL_DOCUMENT")
        qv = embed([q["question"] for q in questions], "RETRIEVAL_QUERY")
        av = embed([q["asr_text"] or " " for q in questions], "RETRIEVAL_QUERY")
        def dense_current(v):
            s = D2 @ v
            order = [SEC_IDS[i] for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)]
            return [o for o in order if SEC_STATUS[o] != "superseded"]
        def bm_top(text):
            s = bm25.get_scores(text.lower().split())
            return [SEC_IDS[i] for i in sorted(range(len(s)), key=lambda i: s[i], reverse=True)]
        gold = {q["id"]: set(re.findall(r"[A-Z]+(?:v\d+)?-\d+", q["gold_evidence"])) & set(SEC_IDS) for q in questions}
        # retrieval metrics for dense_current on the 47 gold questions
        n = 0; hq = ha = hany = 0; mrr = 0.0
        for i, q in enumerate(questions):
            if not gold[q["id"]]:
                continue
            n += 1
            rq = dense_current(qv[i]); ra = dense_current(av[i])
            a = bool(gold[q["id"]] & set(rq[:5])); b = bool(gold[q["id"]] & set(ra[:5]))
            hq += a; ha += b; hany += (a or b)
            for pos, sid in enumerate(rq):
                if sid in gold[q["id"]]:
                    mrr += 1.0 / (pos + 1); break
        STEP["dense_current"] = {"n": n, "hit5_any": hany, "hit5_question": hq, "hit5_asr": ha, "mrr_question": round(mrr / n, 3)}
        tops = {}
        for i, q in enumerate(questions):
            tops[q["id"]] = {"llm_only": [], "rag_bm25": bm_top(q["question"])[:5], "rag_dense": dense_current(qv[i])[:5]}
        def one(args):
            q, arm = args
            sys_p = BASE.format(lang=q["reply_language"] or q["language"])
            ids = tops[q["id"]][arm]
            if arm == "llm_only":
                prompt = "Customer message: " + q["question"]
            else:
                sys_p += RAG_ADD
                prompt = "Policy sections:\n" + ctx_text(ids) + "\n\nCustomer message: " + q["question"]
            out = parse_json(gem(prompt, sys_p, as_json=True))
            reply = str(out.get("reply", "")); dec = str(out.get("decision", "")).lower().strip()
            allowed = numbers(q["question"] + " " + q["reference_answer"] + " " + q["must_include"]) | {"20", "2026"}
            for g in gold[q["id"]]:
                allowed |= numbers(SECTION[g]["body"] + " " + SECTION[g]["title"])
            unsupported = sorted(numbers(reply) - allowed)
            missing, bad = check_must(reply, q["must_include"], q["must_not_include"])
            acc = [s.strip() for s in q["acceptable_decisions"].split(";") if s.strip()] or [q["expected_decision"]]
            return {"id": q["id"], "arm": arm, "answerable": q["answerable"] == "True", "decision": dec, "decision_ok": dec in acc, "facts_ok": (not missing) and (not bad), "missing": missing, "bad": bad, "unsupported": unsupported, "gold_hit": bool(gold[q["id"]] & set(ids)), "reply": reply, "lang": q["language"], "question": q["question"]}
        jobs = [(q, arm) for q in questions for arm in ("llm_only", "rag_bm25", "rag_dense")]
        with ThreadPoolExecutor(4) as ex:
            STEP["rows"] = list(ex.map(one, jobs))
    except Exception as e:
        STEP["err"] = repr(e)
    STEP["done"] = True
threading.Thread(target=run_step_eval, daemon=True).start()
print("step 1 vs step 2 evaluation started")


step 1 vs step 2 evaluation started


In [112]:
print(STEP["done"], STEP["err"], len(STEP.get("rows", [])), "rows")
print("dense_current", STEP.get("dense_current"))
import collections
R = STEP.get("rows", [])
for arm in ("llm_only", "rag_bm25", "rag_dense"):
    A = [r for r in R if r["arm"] == arm]
    ans = [r for r in A if r["answerable"]]
    idk = [r for r in A if not r["answerable"]]
    print(arm, "| all dec_ok", sum(r["decision_ok"] for r in A), "/", len(A), "| facts_ok", sum(r["facts_ok"] for r in A), "/", len(A), "| answerable: dec+facts+no_unsupported", sum(r["decision_ok"] and r["facts_ok"] and not r["unsupported"] for r in ans), "/", len(ans), "| replies w/ unsupported numbers", sum(bool(r["unsupported"]) for r in A), "/", len(A), "| gold retrieved", sum(r["gold_hit"] for r in ans), "/", len(ans), "| IDK ok", sum(r["decision_ok"] and r["facts_ok"] for r in idk), "/", len(idk))

True ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0, description: Embed content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\\nPlease retry in 37.1029447s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per minute limit exceeded on Free Tier. See h

In [237]:
sub=subprocess.run
keys=[os.environ['GNANI_API_KEY'],os.environ['GEMINI_API_KEY']]
hits=[]
for root,_,fs in os.walk('/content/stage/voiceops'):
    for f in fs:
        p=os.path.join(root,f)
        try: b=open(p,'rb').read()
        except: continue
        for k in keys:
            if k.encode() in b: hits.append(p)
print('key hits:',hits)
print(sub('cd /content/stage/voiceops && find . -name ".env" -o -name "*.ipynb" -o -name "venv" -o -name "cache" | head; ls -A',shell=True,capture_output=True,text=True).stdout)
D='/content/drive/MyDrive/Capstone_submission'
sub('cd /content/stage/voiceops && rm -f '+D+'/arvind_voiceops.zip && zip -qr '+D+'/arvind_voiceops.zip . -x ".DS_Store"',shell=True)
print(sub('ls -la '+D+'; unzip -l '+D+'/arvind_voiceops.zip | awk "{print \\$4}" | grep -v "/.*/" | head -30; unzip -l '+D+'/arvind_voiceops.zip | tail -1',shell=True,capture_output=True,text=True).stdout)

key hits: []
.env.example
.gitignore
README.md
report.pdf
requirements.txt
results
run.py
run.sh
sample_inputs
sample_outputs
src

total 16479
drwx------  2 root root    4096 Oct 25 15:30 .
drwx------ 13 root root    4096 Oct 25 10:15 ..
-rw-------  1 root root 8067217 Oct 25 15:30 arvind_voiceops.zip
-rw-------  1 root root  740853 Oct 25 13:12 voiceops_repo_interim.zip
-rw-------  1 root root 8056944 Oct 25 15:20 voiceops_submission_candidate.zip

Name
----
requirements.txt
.env.example
run.py
report.pdf
results/
results/step_comparison.json
results/voice_run2.json
results/retrieval_dense_filtered.json
results/conversation_eval.json
results/voice_first_run.json
results/step_comparison_run1.json
results/conversation_eval_run_a.json
results/voice_run3_final.json
results/retrieval_comparison_notebook.json
run.sh
src/
src/data.py
src/voice_tests.py
src/evaluation.py
src/retrieval.py
src/templates.py
src/pipeline.py
src/text_utils.py
src/policy.py
src/config.py
src/voice.py
src/__init__.p